In [28]:
import os
import json
import zipfile
import numpy as np
import copy
import requests
import io
import shutil

import pandas as pd
import geopandas as gpd

import shapely.wkt as wkt
from shapely.geometry import LineString
from shapely.geometry import Polygon
from shapely.geometry import Point

import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from matplotlib import cm
from matplotlib.colors import LinearSegmentedColormap
from matplotlib.colors import BoundaryNorm

from math import radians, cos, sin, asin, sqrt

In [29]:
DIR="/home/dy23a.fsu/st/datasets/raw"
os.makedirs(DIR, exist_ok=True)
DIR=os.path.join(DIR,"NYC")
os.makedirs(DIR, exist_ok=True)

nyc_shape="./NYC Taxi Zones.geojson"
nyc_shapefile_url="https://d37ci6vzurychx.cloudfront.net/misc/taxi_zones.zip"

taxi_orig_dir=os.path.join(DIR,"taxi")
bike_orig_file=os.path.join(DIR,"bike")

# Download

Taxi 2025 https://www.nyc.gov/site/tlc/about/tlc-trip-record-data.page  
Bike 2025 https://s3.amazonaws.com/tripdata/index.html  
Subway 2024 https://catalog.data.gov/dataset/mta-subway-origin-destination-ridership-estimate-2024  

## 1 Shapefile

In [30]:
if not os.path.exists(nyc_shape):
    response = requests.get(nyc_shapefile_url)
    with zipfile.ZipFile(io.BytesIO(response.content)) as zip_ref:
        zip_ref.extractall("./")
    myshpfile = gpd.read_file('./taxi_zones/taxi_zones.shp')
    myshpfile.to_file(nyc_shape, driver='GeoJSON')
    shutil.rmtree("./taxi_zones")

## 2 Taxi

In [31]:
DOWNLOAD=True
year = 2025
taxi_types = ['yellow', 'green', 'fhvhv', 'fhv'] 
base_url_template = "https://d37ci6vzurychx.cloudfront.net/trip-data/{type}_tripdata_{year}-{month:02d}.parquet"
os.makedirs(taxi_orig_dir, exist_ok=True)
if DOWNLOAD:
    for taxi_type in taxi_types:
        for month in range(1, 13):
            url = base_url_template.format(type=taxi_type, year=year, month=month)
            file_name = f"{taxi_type}_tripdata_{year}-{month:02d}.parquet"
            save_path = os.path.join(taxi_orig_dir, file_name)
            try:
                response = requests.get(url, stream=True, timeout=30)
                if response.status_code == 200:
                    with open(save_path, 'wb') as f:
                        for chunk in response.iter_content(chunk_size=8192 * 1024):
                            if chunk:
                                f.write(chunk)
            except requests.exceptions.RequestException as e:
                print(f"Error: {e}")


## 3 Bike

In [32]:
DOWNLOAD=False
year = 2025
base_url_template = "https://s3.amazonaws.com/tripdata/{year}{month:02d}-citibike-tripdata.zip"
os.makedirs(bike_orig_file, exist_ok=True)
if DOWNLOAD:
    for month in range(1, 13):
        url = base_url_template.format(year=year, month=month)
        zip_filename = f"{year}{month:02d}-citibike-tripdata.zip"
        zip_filepath = os.path.join(bike_orig_file, zip_filename)
        try:
            response = requests.get(url, stream=True, timeout=30)
            
            if response.status_code == 200:
                with open(zip_filepath, 'wb') as f:
                    for chunk in response.iter_content(chunk_size=8192 * 1024): # 每次 8MB
                        if chunk:
                            f.write(chunk)

                with zipfile.ZipFile(zip_filepath, 'r') as zip_ref:
                    zip_ref.extractall(bike_orig_file)
                os.remove(zip_filepath)
                
        except requests.exceptions.RequestException as e:
            print(f"Error: {e}")